In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

# 1. Load everything 
full_grid_encoded = pd.read_csv("../data/taylor_swift_model_df_final_nlp.csv")
model_full_nlp = joblib.load("../data/xgb_model_final_nlp.pkl")

tracklists_df = pd.read_csv("../data/taylor_swift_album_tracklists.csv")
album_songs_lookup = tracklists_df.groupby("album")["track_name"].apply(set).to_dict()

lyric_features_df = pd.read_csv("../data/taylor_swift_lyric_features.csv")

with open("../data/taylor_swift_playcounts.json") as f:
    playcount_scores = json.load(f)
playcount_df = pd.DataFrame([
    {"song_name": song, "playcount": pc} for song, pc in playcount_scores.items()
])

In [ ]:
tour_order = [
    "Fearless", "Speak Now World Tour", "The Red Tour",
    "The 1989 World Tour", "reputation Stadium Tour", "The Eras Tour"
]
next_tour_index = len(tour_order)  # index 6 — one past your last known tour

feature_cols_final = [
    "played_last_tour", "song_age_tours", "tour_type_single_album",
    "sentiment", "avg_sentence_len", "proper_noun_density", "unique_word_ratio",
    "log_playcount"
]

In [ ]:
# 2. Build the prediction input table 
showgirl_songs = album_songs_lookup.get("The Life of a Showgirl", set())
all_known_songs = set(full_grid_encoded["song_name"].unique()) | showgirl_songs

prediction_rows = []
for song in all_known_songs:
    song_history = full_grid_encoded[full_grid_encoded["song_name"] == song].sort_values("tour_index")
    
    if song in showgirl_songs:
        played_last_tour = 0
        song_age_tours = 0
    elif len(song_history) > 0:
        last_row = song_history.iloc[-1]
        played_last_tour = last_row["was_played_on_tour"]
        song_age_tours = next_tour_index - (last_row["tour_index"] - last_row["song_age_tours"])
    else:
        continue
    
    prediction_rows.append({
        "song_name": song,
        "played_last_tour": played_last_tour,
        "song_age_tours": song_age_tours,
        "tour_type_single_album": 1,
    })

prediction_df = pd.DataFrame(prediction_rows)
print(prediction_df.shape)
prediction_df.head()

(501, 4)


,song_name,played_last_tour,song_age_tours,tour_type_single_album
0,Daylight,1,1.0,1
1,Better Man,1,2.0,1
2,There's Nothing Holdin' Me Back,0,2.0,1
3,I Don't Wanna Live Forever,1,2.0,1
4,I'm Only Me When I'm With You,1,6.0,1


In [ ]:
# 3. Merge in lyric (NLP) features 
prediction_df = prediction_df.merge(lyric_features_df, on="song_name", how="left")

for col in ["sentiment", "avg_sentence_len", "proper_noun_density", "unique_word_ratio"]:
    prediction_df[col] = prediction_df[col].fillna(full_grid_encoded[col].mean())

print(prediction_df.isna().sum())

song_name                 0
played_last_tour          0
song_age_tours            0
tour_type_single_album    0
sentiment                 0
avg_sentence_len          0
proper_noun_density       0
unique_word_ratio         0
dtype: int64


In [ ]:
# 4. Merge in popularity (Last.fm playcount)
prediction_df = prediction_df.merge(playcount_df, on="song_name", how="left")
prediction_df["playcount"] = prediction_df["playcount"].fillna(prediction_df["playcount"].median())
prediction_df["log_playcount"] = np.log1p(prediction_df["playcount"])

print(prediction_df.isna().sum())

In [ ]:
# 5. Predict
X_predict = prediction_df[feature_cols_final]
prediction_df["predicted_probability"] = model_full_nlp.predict_proba(X_predict)[:, 1]

top_predictions = prediction_df.sort_values("predicted_probability", ascending=False)
top_predictions.to_csv("../data/showgirl_tour_prediction.csv", index=False)

top_predictions[["song_name", "predicted_probability"]].head(30)

,song_name,predicted_probability
157,All Too Well,0.997435
299,White Horse,0.996380
104,Wi$h Li$t,0.996182
446,Wood,0.996182
359,Honey,0.996182
38,Elizabeth Taylor,0.996182
385,Actually Romantic,0.996182
202,Father Figure,0.996182
120,The Life of a Showgirl (feat. Sabrina Carpenter),0.996182
268,Ruin The Friendship,0.996182
